# Width sweeps on Kaggle

Runs sweeps of `width_sweep.py` (see `docs/WIDTH-QUESTIONS.md`) or `sanford_repro.py`
(the Sanford et al. 2024a reproduction) as parallel jobs on the notebook's GPU(s).

**Settings (right panel):** Accelerator = GPU T4 x2 (recent PyTorch builds dropped P100 support), Internet = On.
Run it unattended with **Save Version → Save & Run All (Commit)**, then download
`<results>_results.zip` (`width_results.zip` or `sanford_results.zip`) from the version's Output tab.

In [ ]:
SCRIPT = "sanford_repro.py"   # or "width_sweep.py"
SWEEPS = ["a1k", "a100k"]     # sweep keys of SCRIPT; each gets SHARDS workers
SHARDS = 1                    # workers per sweep; width_sweep's tiny models share a GPU 8 ways
BRANCH = "master"

import os
os.environ["WIDTH_CACHE"] = "/tmp/width_cache"   # datasets cache, kept out of the output zip
RESULTS = "width" if SCRIPT == "width_sweep.py" else "sanford"


In [ ]:
!git clone -q --depth 1 -b $BRANCH https://github.com/barsboldb/mpgnn.git
%cd mpgnn
!pip install -q torch-geometric
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git log -1 --format='%h %s'


In [ ]:
# Build every dataset the sweeps need (no-op for sweeps without cached data).
for sw in SWEEPS:
    !python $SCRIPT {sw} --prepare


In [ ]:
import os, subprocess, time, torch

gpus = max(1, torch.cuda.device_count())
out = f"results/{RESULTS}"
os.makedirs(out, exist_ok=True)
procs = []
for j, (sw, i) in enumerate((sw, i) for sw in SWEEPS for i in range(SHARDS)):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(j % gpus))
    log = open(f"{out}/{sw}.shard{i}.log", "w")
    procs.append(subprocess.Popen(["python", "-u", SCRIPT, sw, "--shard", f"{i}/{SHARDS}"],
                                  stdout=log, stderr=subprocess.STDOUT, env=env))
print(f"{len(procs)} jobs on {gpus} GPU(s)")

t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(120)
    done = sum(1 for f in os.listdir(out) if f.endswith(".jsonl") and f.split(".")[0] in SWEEPS
               for _ in open(f"{out}/{f}"))
    print(f"{(time.time() - t0) / 60:5.1f} min  runs saved: {done}", flush=True)
print("exit codes:", [p.returncode for p in procs])


In [ ]:
# Catch anything a job missed or crashed on (no-op when complete), then analyze.
for sw in SWEEPS:
    !python -u $SCRIPT {sw}
    !python $SCRIPT {sw} --analyze
    if SCRIPT == "width_sweep.py":
        !python $SCRIPT {sw} --curves


In [ ]:
!cd results && zip -qr /kaggle/working/{RESULTS}_results.zip {RESULTS} && ls -lh /kaggle/working/{RESULTS}_results.zip
